In [ ]:
%pip -q install gradio cmake ninja pyngrok

In [ ]:
!nvidia-smi

In [ ]:
import os

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
os.environ["FORCE_CMAKE"] = "1"

# Install llama-cpp-python using the pre-compiled CUDA wheels and 2 cores for compilation
%pip install --only-binary=:all: --no-cache-dir \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --compile --config-settings=--global-option="--parallel=2"

In [ ]:
from google.colab import drive
import os
import sqlite3

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/LocalLLM"
MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "qwen3-27b",
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-IQ2_M.gguf",
)
CHAT_DB = os.path.join(BASE_DIR, "chat_history.sqlite3")

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found in Google Drive: {MODEL_PATH}")
if not os.path.isfile(CHAT_DB):
    raise FileNotFoundError(f"Chat database not found in Google Drive: {CHAT_DB}")

with sqlite3.connect(CHAT_DB) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

print("Model file:", MODEL_PATH)
print("Chat database:", CHAT_DB)

In [ ]:
import llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python was installed without CUDA support. "
        "Re-run the CUDA installation cell in a GPU-enabled Colab runtime."
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=18432,
    verbose=False,
)
print("Qwen model loaded with GPU offload enabled.")

In [ ]:

import os
import socket
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# ==================================================
# 1. Load credentials from Colab Secrets
# ==================================================

ngrok_token = userdata.get("NGROK_AUTHTOKEN")
ngrok_domain = userdata.get("NGROK_DOMAIN")

if not ngrok_token:
    raise RuntimeError(
        "NGROK_AUTHTOKEN is missing from Colab Secrets."
    )

if not ngrok_domain:
    raise RuntimeError(
        "NGROK_DOMAIN is missing from Colab Secrets."
    )

# Accept either a hostname or a full HTTPS URL.
ngrok_domain = ngrok_domain.strip().rstrip("/")
if ngrok_domain.startswith("https://"):
    ngrok_domain = ngrok_domain[len("https://"):]
elif ngrok_domain.startswith("http://"):
    ngrok_domain = ngrok_domain[len("http://"):]

if "/" in ngrok_domain:
    raise ValueError(
        "NGROK_DOMAIN must contain only the reserved hostname, "
        "without a path."
    )

# ==================================================
# 2. Configure ngrok authentication
# ==================================================

try:
    ngrok.set_auth_token(ngrok_token)
    print("Ngrok authentication configured.")
except Exception as e:
    if "Authtoken already set in configuration file" in str(e):
        print("Using existing ngrok authentication.")
    else:
        raise

# ==================================================
# 3. Check the existing Gradio server
# ==================================================

# Your previous logs showed Gradio running on port 55007.
port = 55007

try:
    with socket.create_connection(
        ("127.0.0.1", port), timeout=5
    ):
        print(f"Gradio is responding on port {port}.")
except OSError as e:
    raise RuntimeError(
        f"Gradio is not responding on port {port}. "
        "Run your existing demo.launch() cell first."
    ) from e

# ==================================================
# 4. Check ngrok agent version
# ==================================================

try:
    result = subprocess.run(
        ["ngrok", "version"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print("Ngrok agent:", result.stdout.strip())
except Exception as e:
    print("Could not check ngrok version:", e)

# ==================================================
# 5. Connect the reserved domain
# ==================================================

# Do not launch Gradio again or kill the agent here.
# Use the URL option supported by recent pyngrok APIs.

try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 55)
    print("AI CHATBOT IS ONLINE")
    print("Stable public URL:", tunnel.public_url)
    print("=" * 55)

except Exception as e:
    print("\nNgrok connection failed.")
    print("Error:", repr(e))
    print(
        "\nThe reserved-domain parameter may not be supported "
        "by your installed pyngrok/agent combination. "
        "Check the installed pyngrok API and ngrok endpoint "
        "configuration before retrying."
    )
    raise
